In [18]:
import pandas as pd
import re
import random

random.seed(42)

INPUT_FILE = "x_raw_menteri_keuangan.csv"
OUTPUT_FILE = "x_dataset_emosi_menkeu_bersih1.csv"

pd.read_csv(
    INPUT_FILE,
    engine="python",
    sep=",",
    quoting=3,   # csv.QUOTE_NONE
    encoding="utf-8",
    on_bad_lines="warn"
)

print("Data berhasil dibaca")
print("Jumlah baris:", len(df))
df


Data berhasil dibaca
Jumlah baris: 480


C:\Users\kevin\AppData\Local\Temp\ipykernel_1776\2479119580.py:10: ParserWarning: Skipping line 7: Expected 4 fields in line 7, saw 5

  pd.read_csv(
C:\Users\kevin\AppData\Local\Temp\ipykernel_1776\2479119580.py:10: ParserWarning: Skipping line 34: Expected 4 fields in line 34, saw 5

  pd.read_csv(
C:\Users\kevin\AppData\Local\Temp\ipykernel_1776\2479119580.py:10: ParserWarning: Skipping line 39: Expected 4 fields in line 39, saw 6

  pd.read_csv(
C:\Users\kevin\AppData\Local\Temp\ipykernel_1776\2479119580.py:10: ParserWarning: Skipping line 70: Expected 4 fields in line 70, saw 6

  pd.read_csv(
C:\Users\kevin\AppData\Local\Temp\ipykernel_1776\2479119580.py:10: ParserWarning: Skipping line 78: Expected 4 fields in line 78, saw 7

  pd.read_csv(
C:\Users\kevin\AppData\Local\Temp\ipykernel_1776\2479119580.py:10: ParserWarning: Skipping line 111: Expected 4 fields in line 111, saw 6

  pd.read_csv(
C:\Users\kevin\AppData\Local\Temp\ipykernel_1776\2479119580.py:10: ParserWarning: Skippi

,id,teks,emosi,label
0,1,bu sri mulyani diganti,negatif,fear
1,3,Aku deg²kan Ketika Sri Mulyani diganti,negatif,fear
2,4,BREAKING NEWS Isu Reshuffle Kabinet Kementrian...,negatif,fear
3,5,Sri Mulyani diganti IHSG langsung merah Asik b...,negatif,fear
4,7,Presiden Prabowo Reshuffle Kabinet Merah Putih...,negatif,fear
...,...,...,...,...
475,996,Kebijakan impor dibahas kembali oleh Kementeri...,negatif,fear
476,997,Dukungan publik terhadap Menteri Keuangan baru,positif,surprise
477,998,Opini positif publik terhadap kinerja Menteri ...,positif,joy
478,999,Rangkuman berita nasional terkait Menteri Keua...,negatif,fear


In [19]:
# Jika kolom tidak bernama id & teks, paksa ambil 2 kolom pertama
df = df.iloc[:, :2]
df.columns = ["id", "teks"]

# buang baris kosong
df.dropna(inplace=True)

print("Struktur kolom:")
df


Struktur kolom:


,id,teks
0,1,bu sri mulyani diganti
1,3,Aku deg²kan Ketika Sri Mulyani diganti
2,4,BREAKING NEWS Isu Reshuffle Kabinet Kementrian...
3,5,Sri Mulyani diganti IHSG langsung merah Asik b...
4,7,Presiden Prabowo Reshuffle Kabinet Merah Putih...
...,...,...
475,996,Kebijakan impor dibahas kembali oleh Kementeri...
476,997,Dukungan publik terhadap Menteri Keuangan baru
477,998,Opini positif publik terhadap kinerja Menteri ...
478,999,Rangkuman berita nasional terkait Menteri Keua...


In [20]:
def clean_text(text):
    text = str(text)
    text = re.sub(r"http\S+|https\S+|www\S+|t\.co\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["teks"] = df["teks"].apply(clean_text)

# hapus teks terlalu pendek
df = df[df["teks"].str.len() > 5]

df


,id,teks
0,1,bu sri mulyani diganti
1,3,Aku deg²kan Ketika Sri Mulyani diganti
2,4,BREAKING NEWS Isu Reshuffle Kabinet Kementrian...
3,5,Sri Mulyani diganti IHSG langsung merah Asik b...
4,7,Presiden Prabowo Reshuffle Kabinet Merah Putih...
...,...,...
475,996,Kebijakan impor dibahas kembali oleh Kementeri...
476,997,Dukungan publik terhadap Menteri Keuangan baru
477,998,Opini positif publik terhadap kinerja Menteri ...
478,999,Rangkuman berita nasional terkait Menteri Keua...


In [21]:
positif_keywords = [
    "optimis", "mantap", "harap", "membaik", "dukung",
    "positif", "apresiasi", "baik", "stabil", "tumbuh",
    "percaya", "harapan", "dukungan", "gebrakan"
]

negatif_keywords = [
    "khawatir", "takut", "ragu", "anjlok", "kacau",
    "buruk", "resiko", "kritik", "demo", "cemas",
    "marah", "gagal", "blunder", "ancam"
]

def assign_emosi_label(text):
    t = text.lower()
    pos = any(k in t for k in positif_keywords)
    neg = any(k in t for k in negatif_keywords)

    if pos and not neg:
        return "positif", random.choice(["joy", "surprise"])
    elif neg and not pos:
        return "negatif", random.choice(["anger", "fear"])
    elif pos and neg:
        return random.choice(["positif", "negatif"]), "surprise"
    else:
        return "negatif", "fear"

df[["emosi", "label"]] = df["teks"].apply(
    lambda x: pd.Series(assign_emosi_label(x))
)

df


,id,teks,emosi,label
0,1,bu sri mulyani diganti,negatif,fear
1,3,Aku deg²kan Ketika Sri Mulyani diganti,negatif,fear
2,4,BREAKING NEWS Isu Reshuffle Kabinet Kementrian...,negatif,fear
3,5,Sri Mulyani diganti IHSG langsung merah Asik b...,negatif,fear
4,7,Presiden Prabowo Reshuffle Kabinet Merah Putih...,negatif,fear
...,...,...,...,...
475,996,Kebijakan impor dibahas kembali oleh Kementeri...,negatif,fear
476,997,Dukungan publik terhadap Menteri Keuangan baru,positif,surprise
477,998,Opini positif publik terhadap kinerja Menteri ...,positif,joy
478,999,Rangkuman berita nasional terkait Menteri Keua...,negatif,fear


In [22]:
df_final = df[["id", "teks", "emosi", "label"]].copy()
df_final["id"] = range(1, len(df_final) + 1)

print("Distribusi emosi:")
print(df_final["emosi"].value_counts())

print("\nDistribusi label:")
print(df_final["label"].value_counts())

df_final.to_csv(OUTPUT_FILE, index=False, encoding="utf-8")
print("CSV berhasil dibuat:", OUTPUT_FILE)


Distribusi emosi:
emosi
negatif    430
positif     50
Name: count, dtype: int64

Distribusi label:
label
fear        408
surprise     29
joy          26
anger        17
Name: count, dtype: int64
CSV berhasil dibuat: x_dataset_emosi_menkeu_bersih1.csv
